In [4]:
# ============================================================
# PHASE 1.2D — ProtBERT TRUNCATED EMBEDDING BASELINE
# Colab CUDA/T4 optimized
# Model: Rostlab/prot_bert
# ============================================================

import os
import re
import gc
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

import torch
from transformers import AutoTokenizer, AutoModel

from sklearn.model_selection import StratifiedKFold, cross_validate, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

from sklearn.ensemble import (
    RandomForestClassifier,
    VotingClassifier,
    StackingClassifier,
    HistGradientBoostingClassifier
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    matthews_corrcoef,
    confusion_matrix,
    make_scorer
)

import joblib

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 150)
pd.set_option("display.max_colwidth", None)

print("--- Hardware Check ---")
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Initial GPU memory allocated:", torch.cuda.memory_allocated() / 1024**2, "MB")
    print("Initial GPU memory reserved:", torch.cuda.memory_reserved() / 1024**2, "MB")

--- Hardware Check ---
Torch version: 2.11.0+cu128
CUDA available: True
Using device: cuda
GPU: NVIDIA L4
Initial GPU memory allocated: 0.0 MB
Initial GPU memory reserved: 0.0 MB


In [2]:
# ============================================================
# GOOGLE DRIVE PATHS
# ============================================================
import os
from pathlib import Path
from google.colab import drive

try:
    drive.mount('/content/drive')
except ValueError:
    print("Drive đã được kết nối từ trước.")

# 2. Định nghĩa lại đường dẫn gốc
PROJECT_DIR = Path("/content/drive/MyDrive/Project_Protein")

# Tự động phát hiện nếu bạn viết hoa chữ 'Model' hoặc viết thường chữ 'model'
actual_model_folder = "model"
if not (PROJECT_DIR / actual_model_folder).exists() and (PROJECT_DIR / "Model").exists():
    actual_model_folder = "Model"
SPLIT_DIR = PROJECT_DIR / "model" / "phase1_protein_baseline" / "splits"
BASE_DIR = PROJECT_DIR / "model" / "phase1_2_protbert_truncated_embedding"

EMBED_DIR = BASE_DIR / "embeddings"
RESULT_DIR = BASE_DIR / "results"
MODEL_DIR = BASE_DIR / "models"
REPORT_DIR = BASE_DIR / "reports"

for folder in [BASE_DIR, EMBED_DIR, RESULT_DIR, MODEL_DIR, REPORT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Split dir:", SPLIT_DIR)
print("Output dir:", BASE_DIR)

# 5. Kiểm tra an toàn để chắc chắn không bị lỗi AssertionError nữa
if not SPLIT_DIR.exists():
    print("\n❌ THÔNG BÁO: Vẫn không thấy thư mục dữ liệu splits.")
    print("Danh sách các mục thực tế đang có trong thư mục gốc dự án của bạn:")
    if PROJECT_DIR.exists():
        print(os.listdir(PROJECT_DIR))
    else:
        print(f"Không tìm thấy cả thư mục gốc: {PROJECT_DIR}")
else:
    print("\n✅ THÀNH CÔNG: Đường dẫn hoàn toàn hợp lệ! Hãy chạy tiếp các ô lệnh sau.")

Mounted at /content/drive
Split dir: /content/drive/MyDrive/Project_Protein/model/phase1_protein_baseline/splits
Output dir: /content/drive/MyDrive/Project_Protein/model/phase1_2_protbert_truncated_embedding

✅ THÀNH CÔNG: Đường dẫn hoàn toàn hợp lệ! Hãy chạy tiếp các ô lệnh sau.


In [5]:
# ============================================================
# LOAD SAVED SPLITS
# ============================================================

train_path = SPLIT_DIR / "train_protein_v1.csv"
val_path = SPLIT_DIR / "val_protein_v1.csv"
test_path = SPLIT_DIR / "test_protein_v1.csv"

assert train_path.exists(), train_path
assert val_path.exists(), val_path
assert test_path.exists(), test_path

train_df = pd.read_csv(train_path)
val_df = pd.read_csv(val_path)
test_df = pd.read_csv(test_path)

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

print("\nTrain labels:")
print(train_df["label"].value_counts())

print("\nValidation labels:")
print(val_df["label"].value_counts())

print("\nTest labels:")
print(test_df["label"].value_counts())

Train: (1274, 35)
Validation: (273, 35)
Test: (273, 35)

Train labels:
label
0    637
1    637
Name: count, dtype: int64

Validation labels:
label
1    137
0    136
Name: count, dtype: int64

Test labels:
label
0    137
1    136
Name: count, dtype: int64


In [6]:
# ============================================================
# CLEAN SEQUENCES
# Keep only 20 standard amino acids for consistency
# ============================================================

STANDARD_AAS = set("ACDEFGHIKLMNPQRSTVWY")

def clean_sequence(seq):
    seq = str(seq).upper()
    seq = "".join([aa for aa in seq if aa in STANDARD_AAS])
    return seq


def prepare_protbert_sequence(seq, max_aa_length=1022):
    """
    ProtBERT tokenizer expects space-separated amino acids.

    Example:
        "MKT..." -> "M K T ..."
    """
    seq = clean_sequence(seq)
    seq = seq[:max_aa_length]
    return " ".join(list(seq))


for split_df in [train_df, val_df, test_df]:
    split_df["sequence_clean"] = split_df["sequence"].apply(clean_sequence)
    split_df["sequence_clean_length"] = split_df["sequence_clean"].str.len()

print("Train length summary:")
display(train_df["sequence_clean_length"].describe())

print("Validation length summary:")
display(val_df["sequence_clean_length"].describe())

print("Test length summary:")
display(test_df["sequence_clean_length"].describe())

Train length summary:


,sequence_clean_length
count,1274.000000
mean,744.512559
std,643.266085
min,41.000000
25%,354.000000
50%,555.000000
75%,926.000000
max,7388.000000


Validation length summary:


,sequence_clean_length
count,273.000000
mean,869.728938
std,2114.423720
min,56.000000
25%,370.000000
50%,576.000000
75%,977.000000
max,34350.000000


Test length summary:


,sequence_clean_length
count,273.000000
mean,774.432234
std,689.578849
min,51.000000
25%,326.000000
50%,574.000000
75%,1035.000000
max,4861.000000


In [7]:
# ============================================================
# LOAD ProtBERT MODEL
# ============================================================

PROTBERT_MODEL_NAME = "Rostlab/prot_bert"
MODEL_SHORT_NAME = "ProtBERT"
REPRESENTATION_NAME = "ProtBERT_mean_pool_truncated_1022"

print("Loading tokenizer:", PROTBERT_MODEL_NAME)
tokenizer = AutoTokenizer.from_pretrained(
    PROTBERT_MODEL_NAME,
    do_lower_case=False
)

print("Loading model:", PROTBERT_MODEL_NAME)

if device.type == "cuda":
    protbert_model = AutoModel.from_pretrained(
        PROTBERT_MODEL_NAME,
        torch_dtype=torch.float16
    )
else:
    protbert_model = AutoModel.from_pretrained(PROTBERT_MODEL_NAME)

protbert_model.to(device)
protbert_model.eval()

print("Model loaded.")
print("Representation:", REPRESENTATION_NAME)
print("Device:", device)

if device.type == "cuda":
    print("GPU memory allocated:", torch.cuda.memory_allocated() / 1024**2, "MB")
    print("GPU memory reserved:", torch.cuda.memory_reserved() / 1024**2, "MB")

Loading tokenizer: Rostlab/prot_bert


config.json:   0%|          | 0.00/361 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/86.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/81.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/1.68G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/487 [00:00<?, ?it/s]

BertModel LOAD REPORT from: Rostlab/prot_bert
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/1.68G [00:00<?, ?B/s]

Model loaded.
Representation: ProtBERT_mean_pool_truncated_1022
Device: cuda
GPU memory allocated: 801.5654296875 MB
GPU memory reserved: 822.0 MB


In [8]:
# ============================================================
# EMBEDDING CONFIG
# ============================================================

MAX_AA_LENGTH = 1022
BATCH_SIZE = 2          # If CUDA OOM, reduce to 1
SAVE_EVERY = 50         # checkpoint every 50 proteins

print("Max amino acid length:", MAX_AA_LENGTH)
print("Batch size:", BATCH_SIZE)
print("Save every:", SAVE_EVERY)

Max amino acid length: 1022
Batch size: 2
Save every: 50


In [9]:
# ============================================================
# BATCH EMBEDDING FUNCTION FOR PROTBERT
# ============================================================

@torch.no_grad()
def embed_protbert_batch_truncated(
    seqs,
    tokenizer,
    model,
    device,
    max_aa_length=1022
):
    """
    Convert a batch of protein sequences into ProtBERT embeddings.

    Policy:
    - Sequences are cleaned to 20 standard amino acids.
    - Each sequence is truncated to max_aa_length.
    - ProtBERT input requires space-separated amino acids.
    - Mean-pool amino-acid token embeddings.
    - Exclude special CLS/SEP tokens from pooling.
    """
    prepared_seqs = [
        prepare_protbert_sequence(seq, max_aa_length=max_aa_length)
        for seq in seqs
    ]

    encoded = tokenizer(
        prepared_seqs,
        return_tensors="pt",
        add_special_tokens=True,
        padding=True,
        truncation=True
    )

    encoded = {k: v.to(device) for k, v in encoded.items()}

    if device.type == "cuda":
        with torch.amp.autocast(device_type="cuda", dtype=torch.float16):
            outputs = model(**encoded)
    else:
        outputs = model(**encoded)

    hidden = outputs.last_hidden_state        # [batch, token_len, hidden_dim]
    attention_mask = encoded["attention_mask"]

    batch_embeddings = []

    for i in range(hidden.shape[0]):
        mask = attention_mask[i].bool()
        valid_positions = torch.where(mask)[0]

        # Exclude special tokens if possible
        if len(valid_positions) > 2:
            aa_positions = valid_positions[1:-1]
        else:
            aa_positions = valid_positions

        aa_hidden = hidden[i, aa_positions, :]
        emb = aa_hidden.mean(dim=0)

        batch_embeddings.append(emb.detach().float().cpu().numpy())

    batch_embeddings = np.vstack(batch_embeddings)

    del encoded, outputs, hidden, attention_mask

    return batch_embeddings

In [10]:
# ============================================================
# ROBUST SPLIT EMBEDDING EXTRACTION WITH RESUME
# ============================================================

def extract_protbert_embeddings_for_split_gpu(
    split_df,
    split_name,
    tokenizer,
    model,
    device,
    output_dir,
    sequence_col="sequence_clean",
    max_aa_length=1022,
    batch_size=2,
    save_every=50
):
    output_dir = Path(output_dir)

    final_embedding_path = output_dir / f"protbert_trunc_{split_name}_embeddings.npy"
    final_metadata_path = output_dir / f"protbert_trunc_{split_name}_metadata.csv"

    partial_embedding_path = output_dir / f"protbert_trunc_{split_name}_embeddings_partial.npy"
    partial_metadata_path = output_dir / f"protbert_trunc_{split_name}_metadata_partial.csv"

    # If final exists, load and return
    if final_embedding_path.exists() and final_metadata_path.exists():
        print(f"[{split_name}] Final embeddings already exist. Loading...")
        embeddings = np.load(final_embedding_path)
        metadata = pd.read_csv(final_metadata_path)
        return embeddings, metadata

    split_df_reset = split_df.reset_index(drop=True)

    embeddings = []
    metadata_records = []
    start_idx = 0

    # Resume from partial if available
    if partial_embedding_path.exists() and partial_metadata_path.exists():
        print(f"[{split_name}] Partial checkpoint found. Resuming...")

        partial_embeddings = np.load(partial_embedding_path)
        partial_metadata = pd.read_csv(partial_metadata_path)

        embeddings = [partial_embeddings[i] for i in range(partial_embeddings.shape[0])]
        metadata_records = partial_metadata.to_dict("records")

        start_idx = len(metadata_records)

        print(f"[{split_name}] Resuming from index {start_idx}")

    start_time = time.time()

    for batch_start in range(start_idx, len(split_df_reset), batch_size):
        batch_end = min(batch_start + batch_size, len(split_df_reset))
        batch_df = split_df_reset.iloc[batch_start:batch_end]

        seqs = batch_df[sequence_col].tolist()

        try:
            batch_embeddings = embed_protbert_batch_truncated(
                seqs=seqs,
                tokenizer=tokenizer,
                model=model,
                device=device,
                max_aa_length=max_aa_length
            )

        except RuntimeError as e:
            if "out of memory" in str(e).lower():
                print(f"CUDA OOM at {split_name} batch {batch_start}:{batch_end}.")
                print("Try reducing BATCH_SIZE to 1.")
                if device.type == "cuda":
                    torch.cuda.empty_cache()
            raise e

        for local_i, (_, row) in enumerate(batch_df.iterrows()):
            seq = row[sequence_col]
            emb = batch_embeddings[local_i]

            embeddings.append(emb)

            metadata_records.append({
                "row_index": int(batch_start + local_i),
                "gene_id": row.get("gene_id", None),
                "gene_symbol": row.get("gene_symbol", None),
                "uniprot_accession": row.get("uniprot_accession", None),
                "label": int(row["label"]),
                "sequence_clean_length": len(seq),
                "truncated": len(seq) > max_aa_length,
                "used_length": min(len(seq), max_aa_length),
                "max_aa_length": max_aa_length,
                "representation": REPRESENTATION_NAME
            })

        processed = len(metadata_records)

        # Checkpoint
        if processed % save_every == 0 or processed == len(split_df_reset):
            current_embeddings = np.vstack(embeddings)
            current_metadata = pd.DataFrame(metadata_records)

            np.save(partial_embedding_path, current_embeddings)
            current_metadata.to_csv(partial_metadata_path, index=False)

            elapsed = time.time() - start_time

            print(
                f"[{split_name}] {processed}/{len(split_df_reset)} proteins "
                f"| embeddings={current_embeddings.shape} "
                f"| elapsed={elapsed/60:.2f} min"
            )

            if device.type == "cuda":
                torch.cuda.empty_cache()

    final_embeddings = np.vstack(embeddings)
    final_metadata = pd.DataFrame(metadata_records)

    np.save(final_embedding_path, final_embeddings)
    final_metadata.to_csv(final_metadata_path, index=False)

    print(f"[{split_name}] Final embeddings shape:", final_embeddings.shape)
    print(f"[{split_name}] Saved:", final_embedding_path)
    print(f"[{split_name}] Metadata saved:", final_metadata_path)

    return final_embeddings, final_metadata

In [11]:
# ============================================================
# DEBUG RUN — 10 PROTEINS
# ============================================================

debug_df = train_df.head(10).copy()

X_debug_protbert, meta_debug_protbert = extract_protbert_embeddings_for_split_gpu(
    split_df=debug_df,
    split_name="debug10",
    tokenizer=tokenizer,
    model=protbert_model,
    device=device,
    output_dir=EMBED_DIR,
    sequence_col="sequence_clean",
    max_aa_length=MAX_AA_LENGTH,
    batch_size=BATCH_SIZE,
    save_every=5
)

print("Debug embedding shape:", X_debug_protbert.shape)
display(meta_debug_protbert)

if device.type == "cuda":
    print("GPU memory allocated:", torch.cuda.memory_allocated() / 1024**2, "MB")
    print("GPU memory reserved:", torch.cuda.memory_reserved() / 1024**2, "MB")

[debug10] 10/10 proteins | embeddings=(10, 1024) | elapsed=0.02 min
[debug10] Final embeddings shape: (10, 1024)
[debug10] Saved: /content/drive/MyDrive/Project_Protein/model/phase1_2_protbert_truncated_embedding/embeddings/protbert_trunc_debug10_embeddings.npy
[debug10] Metadata saved: /content/drive/MyDrive/Project_Protein/model/phase1_2_protbert_truncated_embedding/embeddings/protbert_trunc_debug10_metadata.csv
Debug embedding shape: (10, 1024)


,row_index,gene_id,gene_symbol,uniprot_accession,label,sequence_clean_length,truncated,used_length,max_aa_length,representation
0,0,ENSG00000205155,PSENEN,Q9NZ42,0,101,False,101,1022,ProtBERT_mean_pool_truncated_1022
1,1,ENSG00000164530,PI16,Q6UXB8,1,463,False,463,1022,ProtBERT_mean_pool_truncated_1022
2,2,ENSG00000143167,GPA33,Q99795,0,319,False,319,1022,ProtBERT_mean_pool_truncated_1022
3,3,ENSG00000137691,CFAP300,Q9BRQ4,0,267,False,267,1022,ProtBERT_mean_pool_truncated_1022
4,4,ENSG00000095981,KCNK16,Q96T55,1,309,False,309,1022,ProtBERT_mean_pool_truncated_1022
5,5,ENSG00000213996,TM6SF2,Q9BZW4,1,377,False,377,1022,ProtBERT_mean_pool_truncated_1022
6,6,ENSG00000164823,OSGIN2,Q9Y236,0,505,False,505,1022,ProtBERT_mean_pool_truncated_1022
7,7,ENSG00000185716,MOSMO,Q8NHV5,0,167,False,167,1022,ProtBERT_mean_pool_truncated_1022
8,8,ENSG00000181873,IBA57,Q5T440,0,356,False,356,1022,ProtBERT_mean_pool_truncated_1022
9,9,ENSG00000164647,STEAP1,Q9UHE8,1,339,False,339,1022,ProtBERT_mean_pool_truncated_1022


GPU memory allocated: 811.6474609375 MB
GPU memory reserved: 824.0 MB


In [12]:
# ============================================================
# EXTRACT VALIDATION EMBEDDINGS
# ============================================================

X_val_protbert, meta_val_protbert = extract_protbert_embeddings_for_split_gpu(
    split_df=val_df,
    split_name="val",
    tokenizer=tokenizer,
    model=protbert_model,
    device=device,
    output_dir=EMBED_DIR,
    sequence_col="sequence_clean",
    max_aa_length=MAX_AA_LENGTH,
    batch_size=BATCH_SIZE,
    save_every=SAVE_EVERY
)

[val] 50/273 proteins | embeddings=(50, 1024) | elapsed=0.02 min
[val] 100/273 proteins | embeddings=(100, 1024) | elapsed=0.04 min
[val] 150/273 proteins | embeddings=(150, 1024) | elapsed=0.06 min
[val] 200/273 proteins | embeddings=(200, 1024) | elapsed=0.08 min
[val] 250/273 proteins | embeddings=(250, 1024) | elapsed=0.10 min
[val] 273/273 proteins | embeddings=(273, 1024) | elapsed=0.11 min
[val] Final embeddings shape: (273, 1024)
[val] Saved: /content/drive/MyDrive/Project_Protein/model/phase1_2_protbert_truncated_embedding/embeddings/protbert_trunc_val_embeddings.npy
[val] Metadata saved: /content/drive/MyDrive/Project_Protein/model/phase1_2_protbert_truncated_embedding/embeddings/protbert_trunc_val_metadata.csv


In [13]:
# ============================================================
# EXTRACT TEST EMBEDDINGS
# ============================================================

X_test_protbert, meta_test_protbert = extract_protbert_embeddings_for_split_gpu(
    split_df=test_df,
    split_name="test",
    tokenizer=tokenizer,
    model=protbert_model,
    device=device,
    output_dir=EMBED_DIR,
    sequence_col="sequence_clean",
    max_aa_length=MAX_AA_LENGTH,
    batch_size=BATCH_SIZE,
    save_every=SAVE_EVERY
)

[test] 50/273 proteins | embeddings=(50, 1024) | elapsed=0.02 min
[test] 100/273 proteins | embeddings=(100, 1024) | elapsed=0.04 min
[test] 150/273 proteins | embeddings=(150, 1024) | elapsed=0.06 min
[test] 200/273 proteins | embeddings=(200, 1024) | elapsed=0.08 min
[test] 250/273 proteins | embeddings=(250, 1024) | elapsed=0.10 min
[test] 273/273 proteins | embeddings=(273, 1024) | elapsed=0.11 min
[test] Final embeddings shape: (273, 1024)
[test] Saved: /content/drive/MyDrive/Project_Protein/model/phase1_2_protbert_truncated_embedding/embeddings/protbert_trunc_test_embeddings.npy
[test] Metadata saved: /content/drive/MyDrive/Project_Protein/model/phase1_2_protbert_truncated_embedding/embeddings/protbert_trunc_test_metadata.csv


In [14]:
# ============================================================
# EXTRACT TRAIN EMBEDDINGS
# ============================================================

X_train_protbert, meta_train_protbert = extract_protbert_embeddings_for_split_gpu(
    split_df=train_df,
    split_name="train",
    tokenizer=tokenizer,
    model=protbert_model,
    device=device,
    output_dir=EMBED_DIR,
    sequence_col="sequence_clean",
    max_aa_length=MAX_AA_LENGTH,
    batch_size=BATCH_SIZE,
    save_every=SAVE_EVERY
)

[train] 50/1274 proteins | embeddings=(50, 1024) | elapsed=0.02 min
[train] 100/1274 proteins | embeddings=(100, 1024) | elapsed=0.04 min
[train] 150/1274 proteins | embeddings=(150, 1024) | elapsed=0.06 min
[train] 200/1274 proteins | embeddings=(200, 1024) | elapsed=0.08 min
[train] 250/1274 proteins | embeddings=(250, 1024) | elapsed=0.10 min
[train] 300/1274 proteins | embeddings=(300, 1024) | elapsed=0.12 min
[train] 350/1274 proteins | embeddings=(350, 1024) | elapsed=0.14 min
[train] 400/1274 proteins | embeddings=(400, 1024) | elapsed=0.16 min
[train] 450/1274 proteins | embeddings=(450, 1024) | elapsed=0.18 min
[train] 500/1274 proteins | embeddings=(500, 1024) | elapsed=0.20 min
[train] 550/1274 proteins | embeddings=(550, 1024) | elapsed=0.22 min
[train] 600/1274 proteins | embeddings=(600, 1024) | elapsed=0.24 min
[train] 650/1274 proteins | embeddings=(650, 1024) | elapsed=0.26 min
[train] 700/1274 proteins | embeddings=(700, 1024) | elapsed=0.27 min
[train] 750/1274 prote

In [15]:
# ============================================================
# LOAD SAVED PROTBERT TRUNCATED EMBEDDINGS
# ============================================================

X_train_protbert = np.load(EMBED_DIR / "protbert_trunc_train_embeddings.npy")
X_val_protbert = np.load(EMBED_DIR / "protbert_trunc_val_embeddings.npy")
X_test_protbert = np.load(EMBED_DIR / "protbert_trunc_test_embeddings.npy")

meta_train_protbert = pd.read_csv(EMBED_DIR / "protbert_trunc_train_metadata.csv")
meta_val_protbert = pd.read_csv(EMBED_DIR / "protbert_trunc_val_metadata.csv")
meta_test_protbert = pd.read_csv(EMBED_DIR / "protbert_trunc_test_metadata.csv")

y_train = meta_train_protbert["label"].astype(int)
y_val = meta_val_protbert["label"].astype(int)
y_test = meta_test_protbert["label"].astype(int)

print("X_train_protbert:", X_train_protbert.shape)
print("X_val_protbert:", X_val_protbert.shape)
print("X_test_protbert:", X_test_protbert.shape)

print("\nTrain labels:")
print(y_train.value_counts())

display(meta_train_protbert.head())

X_train_protbert: (1274, 1024)
X_val_protbert: (273, 1024)
X_test_protbert: (273, 1024)

Train labels:
label
0    637
1    637
Name: count, dtype: int64


,row_index,gene_id,gene_symbol,uniprot_accession,label,sequence_clean_length,truncated,used_length,max_aa_length,representation
0,0,ENSG00000205155,PSENEN,Q9NZ42,0,101,False,101,1022,ProtBERT_mean_pool_truncated_1022
1,1,ENSG00000164530,PI16,Q6UXB8,1,463,False,463,1022,ProtBERT_mean_pool_truncated_1022
2,2,ENSG00000143167,GPA33,Q99795,0,319,False,319,1022,ProtBERT_mean_pool_truncated_1022
3,3,ENSG00000137691,CFAP300,Q9BRQ4,0,267,False,267,1022,ProtBERT_mean_pool_truncated_1022
4,4,ENSG00000095981,KCNK16,Q96T55,1,309,False,309,1022,ProtBERT_mean_pool_truncated_1022


In [16]:
# ============================================================
# TRUNCATION SUMMARY
# ============================================================

protbert_truncation_summary = []

for split_name, meta in [
    ("train", meta_train_protbert),
    ("validation", meta_val_protbert),
    ("test", meta_test_protbert)
]:
    protbert_truncation_summary.append({
        "split": split_name,
        "n": len(meta),
        "n_truncated": int(meta["truncated"].sum()),
        "pct_truncated": meta["truncated"].mean() * 100,
        "mean_length": meta["sequence_clean_length"].mean(),
        "median_length": meta["sequence_clean_length"].median(),
        "max_length": meta["sequence_clean_length"].max(),
        "mean_used_length": meta["used_length"].mean()
    })

protbert_truncation_summary_df = pd.DataFrame(protbert_truncation_summary)

display(protbert_truncation_summary_df)

protbert_truncation_summary_df.to_csv(
    RESULT_DIR / "protbert_truncation_summary_v1.csv",
    index=False
)

,split,n,n_truncated,pct_truncated,mean_length,median_length,max_length,mean_used_length
0,train,1274,263,20.643642,744.512559,555.0,7388,605.529827
1,validation,273,65,23.809524,869.728938,576.0,34350,621.208791
2,test,273,69,25.274725,774.432234,574.0,4861,608.311355


In [17]:
# ============================================================
# FREE GPU MEMORY AFTER EMBEDDING
# ============================================================

del protbert_model
torch.cuda.empty_cache()
gc.collect()

print("GPU memory cleared.")

GPU memory cleared.


In [18]:
# ============================================================
# EVALUATION HELPERS
# ============================================================

def get_positive_class_score(model, X):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]

    if hasattr(model, "decision_function"):
        return model.decision_function(X)

    return model.predict(X)


def evaluate_binary_classifier(model, X, y, model_name, dataset_name):
    y_pred = model.predict(X)
    y_score = get_positive_class_score(model, X)

    tn, fp, fn, tp = confusion_matrix(y, y_pred).ravel()

    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    return {
        "model": model_name,
        "dataset": dataset_name,

        "accuracy": accuracy_score(y, y_pred),
        "precision": precision_score(y, y_pred, zero_division=0),
        "recall_sensitivity": recall_score(y, y_pred, zero_division=0),
        "specificity": specificity,
        "f1": f1_score(y, y_pred, zero_division=0),

        "roc_auc": roc_auc_score(y, y_score),
        "pr_auc": average_precision_score(y, y_score),
        "mcc": matthews_corrcoef(y, y_pred),

        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp
    }

In [19]:
# ============================================================
# CV + SCORING
# ============================================================

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision",
    "mcc": make_scorer(matthews_corrcoef)
}

In [20]:
# ============================================================
# DOWNSTREAM MODELS FOR PROTBERT EMBEDDINGS
# ============================================================

try:
    from lightgbm import LGBMClassifier
    LIGHTGBM_AVAILABLE = True
except ImportError:
    LIGHTGBM_AVAILABLE = False


protbert_models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            max_iter=5000,
            random_state=42
        ))
    ]),

    "SVM RBF": Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVC(
            kernel="rbf",
            probability=True,
            random_state=42
        ))
    ]),

    "Random Forest": Pipeline([
        ("model", RandomForestClassifier(
            n_estimators=300,
            random_state=42,
            n_jobs=-1
        ))
    ])
}


if LIGHTGBM_AVAILABLE:
    protbert_models["LightGBM"] = Pipeline([
        ("model", LGBMClassifier(
            n_estimators=300,
            learning_rate=0.05,
            num_leaves=15,
            random_state=42,
            n_jobs=-1,
            verbose=-1
        ))
    ])
else:
    protbert_models["Hist Gradient Boosting"] = Pipeline([
        ("model", HistGradientBoostingClassifier(
            learning_rate=0.05,
            max_iter=300,
            random_state=42
        ))
    ])

print("Models:")
for name in protbert_models:
    print("-", name)

Models:
- Logistic Regression
- SVM RBF
- Random Forest
- LightGBM


In [21]:
# ============================================================
# BASELINE CV BEFORE TUNING
# ============================================================

protbert_baseline_cv_results = []

for model_name, pipeline in protbert_models.items():
    print("=" * 80)
    print("ProtBERT baseline CV:", model_name)

    cv_output = cross_validate(
        estimator=pipeline,
        X=X_train_protbert,
        y=y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1,
        return_train_score=True
    )

    row = {
        "representation": REPRESENTATION_NAME,
        "model": model_name,

        "train_roc_auc_mean": cv_output["train_roc_auc"].mean(),
        "train_roc_auc_std": cv_output["train_roc_auc"].std(),

        "cv_roc_auc_mean": cv_output["test_roc_auc"].mean(),
        "cv_roc_auc_std": cv_output["test_roc_auc"].std(),

        "cv_f1_mean": cv_output["test_f1"].mean(),
        "cv_f1_std": cv_output["test_f1"].std(),

        "cv_pr_auc_mean": cv_output["test_pr_auc"].mean(),
        "cv_pr_auc_std": cv_output["test_pr_auc"].std(),

        "cv_mcc_mean": cv_output["test_mcc"].mean(),
        "cv_mcc_std": cv_output["test_mcc"].std(),

        "overfit_gap_train_minus_cv": (
            cv_output["train_roc_auc"].mean() - cv_output["test_roc_auc"].mean()
        )
    }

    protbert_baseline_cv_results.append(row)

protbert_baseline_cv_df = pd.DataFrame(protbert_baseline_cv_results).sort_values(
    by="cv_roc_auc_mean",
    ascending=False
)

display(protbert_baseline_cv_df)

protbert_baseline_cv_df.to_csv(
    RESULT_DIR / "protbert_baseline_cv_before_tuning_v1.csv",
    index=False
)

ProtBERT baseline CV: Logistic Regression
ProtBERT baseline CV: SVM RBF
ProtBERT baseline CV: Random Forest
ProtBERT baseline CV: LightGBM


,representation,model,train_roc_auc_mean,train_roc_auc_std,cv_roc_auc_mean,cv_roc_auc_std,cv_f1_mean,cv_f1_std,cv_pr_auc_mean,cv_pr_auc_std,cv_mcc_mean,cv_mcc_std,overfit_gap_train_minus_cv
1,ProtBERT_mean_pool_truncated_1022,SVM RBF,0.942903,2.526241e-03,0.720084,0.032127,0.656671,0.025527,0.714467,0.033381,0.305604,0.065856,0.222819
2,ProtBERT_mean_pool_truncated_1022,Random Forest,1.000000,4.965068e-17,0.716401,0.029192,0.653287,0.015713,0.714343,0.030052,0.299591,0.044666,0.283599
3,ProtBERT_mean_pool_truncated_1022,LightGBM,1.000000,0.000000e+00,0.696829,0.034818,0.638282,0.021358,0.697256,0.028334,0.275685,0.060473,0.303171
0,ProtBERT_mean_pool_truncated_1022,Logistic Regression,0.999999,1.540891e-06,0.645658,0.033915,0.619355,0.022187,0.639820,0.049640,0.229727,0.036464,0.354342


In [22]:
# ============================================================
# PARAMETER GRIDS — COLAB OPTIMIZED
# ============================================================

protbert_param_grids = {
    "Logistic Regression": {
        "model__C": [0.0003, 0.001, 0.003, 0.01],
        "model__penalty": ["l2"],
        "model__solver": ["lbfgs"]
    },

    "SVM RBF": {
        "model__C": [0.1, 1, 10],
        "model__gamma": [0.0001, 0.001, "scale"]
    },

    "Random Forest": {
        "model__n_estimators": [300, 500],
        "model__max_depth": [5, 8, 10],
        "model__min_samples_leaf": [5, 10, 20],
        "model__max_features": ["sqrt", 0.2],
        "model__bootstrap": [True]
    }
}


if LIGHTGBM_AVAILABLE:
    protbert_param_grids["LightGBM"] = {
        "model__n_estimators": [100, 200],
        "model__learning_rate": [0.03, 0.05],
        "model__num_leaves": [7, 15],
        "model__max_depth": [3, 5],
        "model__min_child_samples": [20, 50],
        "model__subsample": [0.8],
        "model__colsample_bytree": [0.8, 1.0],
        "model__reg_alpha": [0.1],
        "model__reg_lambda": [1, 5]
    }
else:
    protbert_param_grids["Hist Gradient Boosting"] = {
        "model__learning_rate": [0.03, 0.05],
        "model__max_iter": [100, 200],
        "model__max_leaf_nodes": [15, 31],
        "model__min_samples_leaf": [20, 50]
    }

In [23]:
# ============================================================
# RUN GRIDSEARCHCV
# ============================================================

protbert_grid_search_results = []
protbert_best_estimators = {}

for model_name, pipeline in protbert_models.items():
    print("=" * 80)
    print("ProtBERT GridSearchCV:", model_name)

    grid = GridSearchCV(
        estimator=pipeline,
        param_grid=protbert_param_grids[model_name],
        scoring="roc_auc",
        cv=cv,
        n_jobs=-1,
        refit=True,
        return_train_score=True,
        verbose=1
    )

    grid.fit(X_train_protbert, y_train)

    protbert_best_estimators[model_name] = grid.best_estimator_

    best_idx = grid.best_index_

    result_row = {
        "representation": REPRESENTATION_NAME,
        "model": model_name,
        "best_cv_roc_auc": grid.best_score_,
        "best_train_roc_auc": grid.cv_results_["mean_train_score"][best_idx],
        "overfit_gap_train_minus_cv": (
            grid.cv_results_["mean_train_score"][best_idx] - grid.best_score_
        ),
        "best_params": grid.best_params_
    }

    protbert_grid_search_results.append(result_row)

    print("Best CV ROC-AUC:", grid.best_score_)
    print("Best train ROC-AUC:", grid.cv_results_["mean_train_score"][best_idx])
    print("Gap:", result_row["overfit_gap_train_minus_cv"])
    print("Best params:", grid.best_params_)

protbert_grid_results_df = pd.DataFrame(protbert_grid_search_results).sort_values(
    by="best_cv_roc_auc",
    ascending=False
)

display(protbert_grid_results_df)

protbert_grid_results_df.to_csv(
    RESULT_DIR / "protbert_gridsearch_results_v1.csv",
    index=False
)

ProtBERT GridSearchCV: Logistic Regression
Fitting 5 folds for each of 4 candidates, totalling 20 fits
Best CV ROC-AUC: 0.73155660998822
Best train ROC-AUC: 0.8163257036548585
Gap: 0.08476909366663843
Best params: {'model__C': 0.001, 'model__penalty': 'l2', 'model__solver': 'lbfgs'}
ProtBERT GridSearchCV: SVM RBF
Fitting 5 folds for each of 9 candidates, totalling 45 fits
Best CV ROC-AUC: 0.7200839714179428
Best train ROC-AUC: 0.9429032433498072
Gap: 0.22281927193186446
Best params: {'model__C': 1, 'model__gamma': 'scale'}
ProtBERT GridSearchCV: Random Forest
Fitting 5 folds for each of 36 candidates, totalling 180 fits
Best CV ROC-AUC: 0.7205974099448198
Best train ROC-AUC: 0.9987833680411569
Gap: 0.27818595809633706
Best params: {'model__bootstrap': True, 'model__max_depth': 8, 'model__max_features': 0.2, 'model__min_samples_leaf': 5, 'model__n_estimators': 300}
ProtBERT GridSearchCV: LightGBM
Fitting 5 folds for each of 128 candidates, totalling 640 fits
Best CV ROC-AUC: 0.716740324

,representation,model,best_cv_roc_auc,best_train_roc_auc,overfit_gap_train_minus_cv,best_params
0,ProtBERT_mean_pool_truncated_1022,Logistic Regression,0.731557,0.816326,0.084769,"{'model__C': 0.001, 'model__penalty': 'l2', 'model__solver': 'lbfgs'}"
2,ProtBERT_mean_pool_truncated_1022,Random Forest,0.720597,0.998783,0.278186,"{'model__bootstrap': True, 'model__max_depth': 8, 'model__max_features': 0.2, 'model__min_samples_leaf': 5, 'model__n_estimators': 300}"
1,ProtBERT_mean_pool_truncated_1022,SVM RBF,0.720084,0.942903,0.222819,"{'model__C': 1, 'model__gamma': 'scale'}"
3,ProtBERT_mean_pool_truncated_1022,LightGBM,0.716740,0.999262,0.282521,"{'model__colsample_bytree': 1.0, 'model__learning_rate': 0.03, 'model__max_depth': 5, 'model__min_child_samples': 50, 'model__n_estimators': 200, 'model__num_leaves': 15, 'model__reg_alpha': 0.1, 'model__reg_lambda': 1, 'model__subsample': 0.8}"


In [24]:
# ============================================================
# VALIDATION EVALUATION
# ============================================================

protbert_tuned_eval_records = []

for model_name, model in protbert_best_estimators.items():
    print("=" * 80)
    print("Evaluating:", model_name)

    train_metrics = evaluate_binary_classifier(
        model=model,
        X=X_train_protbert,
        y=y_train,
        model_name=model_name,
        dataset_name="train"
    )

    val_metrics = evaluate_binary_classifier(
        model=model,
        X=X_val_protbert,
        y=y_val,
        model_name=model_name,
        dataset_name="validation"
    )

    train_metrics["representation"] = REPRESENTATION_NAME
    val_metrics["representation"] = REPRESENTATION_NAME

    protbert_tuned_eval_records.extend([train_metrics, val_metrics])

protbert_tuned_eval_df = pd.DataFrame(protbert_tuned_eval_records)

display(protbert_tuned_eval_df)

protbert_validation_comparison = protbert_tuned_eval_df[
    protbert_tuned_eval_df["dataset"] == "validation"
].sort_values(
    by=["roc_auc", "mcc", "f1"],
    ascending=False
)

display(protbert_validation_comparison)

protbert_tuned_eval_df.to_csv(
    RESULT_DIR / "protbert_tuned_train_validation_metrics_v1.csv",
    index=False
)

protbert_validation_comparison.to_csv(
    RESULT_DIR / "protbert_validation_comparison_v1.csv",
    index=False
)

Evaluating: Logistic Regression
Evaluating: SVM RBF
Evaluating: Random Forest
Evaluating: LightGBM


,model,dataset,accuracy,precision,recall_sensitivity,specificity,f1,roc_auc,pr_auc,mcc,tn,fp,fn,tp,representation
0,Logistic Regression,train,0.731554,0.733017,0.728414,0.734694,0.730709,0.812297,0.813233,0.463117,468,169,173,464,ProtBERT_mean_pool_truncated_1022
1,Logistic Regression,validation,0.681319,0.704918,0.627737,0.735294,0.664093,0.746136,0.715048,0.365095,100,36,51,86,ProtBERT_mean_pool_truncated_1022
2,SVM RBF,train,0.861068,0.873377,0.844584,0.877551,0.858739,0.937763,0.939063,0.722528,559,78,99,538,ProtBERT_mean_pool_truncated_1022
3,SVM RBF,validation,0.695971,0.721311,0.642336,0.750000,0.679537,0.729712,0.690116,0.394566,102,34,49,88,ProtBERT_mean_pool_truncated_1022
4,Random Forest,train,0.981162,0.984202,0.978022,0.984301,0.981102,0.997309,0.997573,0.962342,627,10,14,623,ProtBERT_mean_pool_truncated_1022
5,Random Forest,validation,0.695971,0.714286,0.656934,0.735294,0.684411,0.738729,0.696662,0.393391,100,36,47,90,ProtBERT_mean_pool_truncated_1022
6,LightGBM,train,0.985871,0.987402,0.984301,0.987441,0.985849,0.997826,0.997909,0.971747,629,8,10,627,ProtBERT_mean_pool_truncated_1022
7,LightGBM,validation,0.663004,0.671756,0.642336,0.683824,0.656716,0.731269,0.701433,0.326422,93,43,49,88,ProtBERT_mean_pool_truncated_1022


,model,dataset,accuracy,precision,recall_sensitivity,specificity,f1,roc_auc,pr_auc,mcc,tn,fp,fn,tp,representation
1,Logistic Regression,validation,0.681319,0.704918,0.627737,0.735294,0.664093,0.746136,0.715048,0.365095,100,36,51,86,ProtBERT_mean_pool_truncated_1022
5,Random Forest,validation,0.695971,0.714286,0.656934,0.735294,0.684411,0.738729,0.696662,0.393391,100,36,47,90,ProtBERT_mean_pool_truncated_1022
7,LightGBM,validation,0.663004,0.671756,0.642336,0.683824,0.656716,0.731269,0.701433,0.326422,93,43,49,88,ProtBERT_mean_pool_truncated_1022
3,SVM RBF,validation,0.695971,0.721311,0.642336,0.750000,0.679537,0.729712,0.690116,0.394566,102,34,49,88,ProtBERT_mean_pool_truncated_1022


In [25]:
# ============================================================
# SOFT VOTING
# ============================================================

protbert_voting_estimators = []

for model_name, estimator in protbert_best_estimators.items():
    short_name = model_name.lower().replace(" ", "_").replace("-", "_")
    protbert_voting_estimators.append((short_name, estimator))

protbert_soft_voting = VotingClassifier(
    estimators=protbert_voting_estimators,
    voting="soft",
    n_jobs=-1
)

protbert_soft_voting.fit(X_train_protbert, y_train)

protbert_voting_train_metrics = evaluate_binary_classifier(
    protbert_soft_voting,
    X_train_protbert,
    y_train,
    "Soft Voting",
    "train"
)

protbert_voting_val_metrics = evaluate_binary_classifier(
    protbert_soft_voting,
    X_val_protbert,
    y_val,
    "Soft Voting",
    "validation"
)

protbert_voting_train_metrics["representation"] = REPRESENTATION_NAME
protbert_voting_val_metrics["representation"] = REPRESENTATION_NAME

display(pd.DataFrame([protbert_voting_train_metrics, protbert_voting_val_metrics]))

protbert_best_estimators["Soft Voting"] = protbert_soft_voting

,model,dataset,accuracy,precision,recall_sensitivity,specificity,f1,roc_auc,pr_auc,mcc,tn,fp,fn,tp,representation
0,Soft Voting,train,0.924647,0.934189,0.913658,0.935636,0.923810,0.978781,0.980099,0.849499,596,41,55,582,ProtBERT_mean_pool_truncated_1022
1,Soft Voting,validation,0.688645,0.706349,0.649635,0.727941,0.676806,0.745867,0.700381,0.378696,99,37,48,89,ProtBERT_mean_pool_truncated_1022


In [26]:
# ============================================================
# STACKING
# ============================================================

protbert_stacking_estimators = []

for model_name, estimator in protbert_best_estimators.items():
    if model_name == "Soft Voting":
        continue

    short_name = model_name.lower().replace(" ", "_").replace("-", "_")
    protbert_stacking_estimators.append((short_name, estimator))

protbert_stacking = StackingClassifier(
    estimators=protbert_stacking_estimators,
    final_estimator=LogisticRegression(
        max_iter=5000,
        C=1.0,
        random_state=42
    ),
    cv=5,
    stack_method="predict_proba",
    n_jobs=-1,
    passthrough=False
)

protbert_stacking.fit(X_train_protbert, y_train)

protbert_stacking_train_metrics = evaluate_binary_classifier(
    protbert_stacking,
    X_train_protbert,
    y_train,
    "Stacking",
    "train"
)

protbert_stacking_val_metrics = evaluate_binary_classifier(
    protbert_stacking,
    X_val_protbert,
    y_val,
    "Stacking",
    "validation"
)

protbert_stacking_train_metrics["representation"] = REPRESENTATION_NAME
protbert_stacking_val_metrics["representation"] = REPRESENTATION_NAME

display(pd.DataFrame([protbert_stacking_train_metrics, protbert_stacking_val_metrics]))

protbert_best_estimators["Stacking"] = protbert_stacking

,model,dataset,accuracy,precision,recall_sensitivity,specificity,f1,roc_auc,pr_auc,mcc,tn,fp,fn,tp,representation
0,Stacking,train,0.883046,0.884858,0.880691,0.885400,0.882769,0.949858,0.952773,0.766100,564,73,76,561,ProtBERT_mean_pool_truncated_1022
1,Stacking,validation,0.688645,0.709677,0.642336,0.735294,0.674330,0.745921,0.708297,0.379221,100,36,49,88,ProtBERT_mean_pool_truncated_1022


In [27]:
# ============================================================
# FINAL VALIDATION COMPARISON
# ============================================================

protbert_all_eval_df = pd.concat(
    [
        protbert_tuned_eval_df,
        pd.DataFrame([
            protbert_voting_train_metrics,
            protbert_voting_val_metrics,
            protbert_stacking_train_metrics,
            protbert_stacking_val_metrics
        ])
    ],
    ignore_index=True
)

protbert_all_validation_results = protbert_all_eval_df[
    protbert_all_eval_df["dataset"] == "validation"
].sort_values(
    by=["roc_auc", "mcc", "f1"],
    ascending=False
)

display(protbert_all_validation_results)

protbert_all_eval_df.to_csv(
    RESULT_DIR / "protbert_all_train_validation_metrics_v1.csv",
    index=False
)

protbert_all_validation_results.to_csv(
    RESULT_DIR / "protbert_all_validation_comparison_v1.csv",
    index=False
)

,model,dataset,accuracy,precision,recall_sensitivity,specificity,f1,roc_auc,pr_auc,mcc,tn,fp,fn,tp,representation
1,Logistic Regression,validation,0.681319,0.704918,0.627737,0.735294,0.664093,0.746136,0.715048,0.365095,100,36,51,86,ProtBERT_mean_pool_truncated_1022
11,Stacking,validation,0.688645,0.709677,0.642336,0.735294,0.674330,0.745921,0.708297,0.379221,100,36,49,88,ProtBERT_mean_pool_truncated_1022
9,Soft Voting,validation,0.688645,0.706349,0.649635,0.727941,0.676806,0.745867,0.700381,0.378696,99,37,48,89,ProtBERT_mean_pool_truncated_1022
5,Random Forest,validation,0.695971,0.714286,0.656934,0.735294,0.684411,0.738729,0.696662,0.393391,100,36,47,90,ProtBERT_mean_pool_truncated_1022
7,LightGBM,validation,0.663004,0.671756,0.642336,0.683824,0.656716,0.731269,0.701433,0.326422,93,43,49,88,ProtBERT_mean_pool_truncated_1022
3,SVM RBF,validation,0.695971,0.721311,0.642336,0.750000,0.679537,0.729712,0.690116,0.394566,102,34,49,88,ProtBERT_mean_pool_truncated_1022


In [28]:
# ============================================================
# SELECT FINAL MODEL FROM VALIDATION
# ============================================================

protbert_best_validation_row = protbert_all_validation_results.iloc[0]
final_protbert_model_name = protbert_best_validation_row["model"]
final_protbert_model = protbert_best_estimators[final_protbert_model_name]

print("Final selected ProtBERT model:", final_protbert_model_name)
display(protbert_best_validation_row)

pd.DataFrame([protbert_best_validation_row]).to_csv(
    RESULT_DIR / "protbert_final_model_validation_summary_v1.csv",
    index=False
)

Final selected ProtBERT model: Logistic Regression


,1
model,Logistic Regression
dataset,validation
accuracy,0.681319
precision,0.704918
recall_sensitivity,0.627737
specificity,0.735294
f1,0.664093
roc_auc,0.746136
pr_auc,0.715048
mcc,0.365095


In [29]:
# ============================================================
# FINAL TEST EVALUATION
# ============================================================

protbert_final_test_metrics = evaluate_binary_classifier(
    model=final_protbert_model,
    X=X_test_protbert,
    y=y_test,
    model_name=final_protbert_model_name,
    dataset_name="test"
)

protbert_final_test_metrics["representation"] = REPRESENTATION_NAME

protbert_final_test_metrics_df = pd.DataFrame([protbert_final_test_metrics])

display(protbert_final_test_metrics_df)

protbert_final_test_metrics_df.to_csv(
    RESULT_DIR / "protbert_final_test_metrics_v1.csv",
    index=False
)

,model,dataset,accuracy,precision,recall_sensitivity,specificity,f1,roc_auc,pr_auc,mcc,tn,fp,fn,tp,representation
0,Logistic Regression,test,0.59707,0.592857,0.610294,0.583942,0.601449,0.637076,0.642279,0.194298,80,57,53,83,ProtBERT_mean_pool_truncated_1022


In [30]:
# ============================================================
# DIAGNOSTIC TEST ALL MODELS
# Not used for final model selection.
# ============================================================

protbert_diagnostic_test_records = []

for model_name, model in protbert_best_estimators.items():
    metrics = evaluate_binary_classifier(
        model=model,
        X=X_test_protbert,
        y=y_test,
        model_name=model_name,
        dataset_name="test_diagnostic"
    )

    metrics["representation"] = REPRESENTATION_NAME
    protbert_diagnostic_test_records.append(metrics)

protbert_diagnostic_test_df = pd.DataFrame(protbert_diagnostic_test_records).sort_values(
    by=["roc_auc", "mcc", "f1"],
    ascending=False
)

display(protbert_diagnostic_test_df)

protbert_diagnostic_test_df.to_csv(
    RESULT_DIR / "protbert_diagnostic_all_models_test_metrics_v1.csv",
    index=False
)

,model,dataset,accuracy,precision,recall_sensitivity,specificity,f1,roc_auc,pr_auc,mcc,tn,fp,fn,tp,representation
1,SVM RBF,test_diagnostic,0.597070,0.591549,0.617647,0.576642,0.604317,0.649850,0.647417,0.194446,79,58,52,84,ProtBERT_mean_pool_truncated_1022
5,Stacking,test_diagnostic,0.578755,0.573427,0.602941,0.554745,0.587814,0.639706,0.646704,0.157864,76,61,54,82,ProtBERT_mean_pool_truncated_1022
0,Logistic Regression,test_diagnostic,0.597070,0.592857,0.610294,0.583942,0.601449,0.637076,0.642279,0.194298,80,57,53,83,ProtBERT_mean_pool_truncated_1022
4,Soft Voting,test_diagnostic,0.589744,0.583333,0.617647,0.562044,0.600000,0.636647,0.644129,0.179962,77,60,52,84,ProtBERT_mean_pool_truncated_1022
3,LightGBM,test_diagnostic,0.593407,0.587413,0.617647,0.569343,0.602151,0.623873,0.628893,0.187201,78,59,52,84,ProtBERT_mean_pool_truncated_1022
2,Random Forest,test_diagnostic,0.567766,0.562500,0.595588,0.540146,0.578571,0.620331,0.627068,0.135939,74,63,55,81,ProtBERT_mean_pool_truncated_1022


In [31]:
# ============================================================
# SAVE FINAL TEST PREDICTIONS
# ============================================================

y_test_pred = final_protbert_model.predict(X_test_protbert)
y_test_score = get_positive_class_score(final_protbert_model, X_test_protbert)

protbert_test_predictions_df = meta_test_protbert.copy()
protbert_test_predictions_df["true_label"] = y_test.values
protbert_test_predictions_df["pred_label"] = y_test_pred
protbert_test_predictions_df["pred_score_t2d_associated"] = y_test_score

display(protbert_test_predictions_df.head())

protbert_test_predictions_df.to_csv(
    RESULT_DIR / "protbert_final_test_predictions_v1.csv",
    index=False
)

,row_index,gene_id,gene_symbol,uniprot_accession,label,sequence_clean_length,truncated,used_length,max_aa_length,representation,true_label,pred_label,pred_score_t2d_associated
0,0,ENSG00000177542,SLC25A22,Q9H936,0,323,False,323,1022,ProtBERT_mean_pool_truncated_1022,0,1,0.767247
1,1,ENSG00000123080,CDKN2C,P42773,1,168,False,168,1022,ProtBERT_mean_pool_truncated_1022,1,0,0.451194
2,2,ENSG00000185262,UBALD2,Q8IYN6,0,164,False,164,1022,ProtBERT_mean_pool_truncated_1022,0,1,0.765832
3,3,ENSG00000092148,HECTD1,Q9ULT8,0,2610,True,1022,1022,ProtBERT_mean_pool_truncated_1022,0,0,0.465261
4,4,ENSG00000139364,TMEM132B,Q14DG7,1,1078,True,1022,1022,ProtBERT_mean_pool_truncated_1022,1,1,0.543085


In [32]:
# ============================================================
# SAVE MODELS
# ============================================================

for model_name, model in protbert_best_estimators.items():
    safe_name = model_name.lower().replace(" ", "_").replace("-", "_")

    model_path = MODEL_DIR / f"protbert_{safe_name}_best_estimator_v1.pkl"
    joblib.dump(model, model_path)

    print("Saved:", model_path)

final_protbert_model_path = MODEL_DIR / "protbert_final_selected_model_v1.pkl"
joblib.dump(final_protbert_model, final_protbert_model_path)

print("Final ProtBERT model saved:", final_protbert_model_path)

Saved: /content/drive/MyDrive/Project_Protein/model/phase1_2_protbert_truncated_embedding/models/protbert_logistic_regression_best_estimator_v1.pkl
Saved: /content/drive/MyDrive/Project_Protein/model/phase1_2_protbert_truncated_embedding/models/protbert_svm_rbf_best_estimator_v1.pkl
Saved: /content/drive/MyDrive/Project_Protein/model/phase1_2_protbert_truncated_embedding/models/protbert_random_forest_best_estimator_v1.pkl
Saved: /content/drive/MyDrive/Project_Protein/model/phase1_2_protbert_truncated_embedding/models/protbert_lightgbm_best_estimator_v1.pkl
Saved: /content/drive/MyDrive/Project_Protein/model/phase1_2_protbert_truncated_embedding/models/protbert_soft_voting_best_estimator_v1.pkl
Saved: /content/drive/MyDrive/Project_Protein/model/phase1_2_protbert_truncated_embedding/models/protbert_stacking_best_estimator_v1.pkl
Final ProtBERT model saved: /content/drive/MyDrive/Project_Protein/model/phase1_2_protbert_truncated_embedding/models/protbert_final_selected_model_v1.pkl


In [33]:
# ============================================================
# MASTER COMPARISON TABLE
# Fill previous numbers from your completed phases
# ============================================================

master_rows = [
    {
        "phase": "1.1",
        "representation": "AAC + Physchem",
        "embedding_policy": "handcrafted",
        "final_model": "Random Forest",
        "test_roc_auc": 0.5520,
        "test_pr_auc": 0.5550,
        "test_f1": 0.5390,
        "test_mcc": 0.0480
    },
    {
        "phase": "1.2A",
        "representation": "ESM2_t6_8M",
        "embedding_policy": "truncated_1022",
        "final_model": "Stacking",
        "test_roc_auc": 0.6202,
        "test_pr_auc": 0.6188,
        "test_f1": 0.5926,
        "test_mcc": 0.1941
    },
    {
        "phase": "1.2B",
        "representation": "ESM2_t6_8M",
        "embedding_policy": "sliding_window_1022_stride_1022",
        "final_model": "Soft Voting",
        "test_roc_auc": 0.6277,
        "test_pr_auc": 0.6278,
        "test_f1": 0.5870,
        "test_mcc": 0.1650
    },
    {
        "phase": "1.2C",
        "representation": "ESM2_t12_35M",
        "embedding_policy": "truncated_1022",
        "final_model": "Soft Voting",
        "test_roc_auc": 0.5875,
        "test_pr_auc": 0.5942,
        "test_f1": 0.5654,
        "test_mcc": 0.0995
    },
    {
        "phase": "1.2D",
        "representation": "ProtBERT",
        "embedding_policy": "truncated_1022",
        "final_model": final_protbert_model_name,
        "test_roc_auc": protbert_final_test_metrics["roc_auc"],
        "test_pr_auc": protbert_final_test_metrics["pr_auc"],
        "test_f1": protbert_final_test_metrics["f1"],
        "test_mcc": protbert_final_test_metrics["mcc"]
    }
]

master_comparison_df = pd.DataFrame(master_rows)

display(master_comparison_df)

master_comparison_df.to_csv(
    RESULT_DIR / "master_comparison_with_protbert_v1.csv",
    index=False
)

,phase,representation,embedding_policy,final_model,test_roc_auc,test_pr_auc,test_f1,test_mcc
0,1.1,AAC + Physchem,handcrafted,Random Forest,0.552000,0.555000,0.539000,0.048000
1,1.2A,ESM2_t6_8M,truncated_1022,Stacking,0.620200,0.618800,0.592600,0.194100
2,1.2B,ESM2_t6_8M,sliding_window_1022_stride_1022,Soft Voting,0.627700,0.627800,0.587000,0.165000
3,1.2C,ESM2_t12_35M,truncated_1022,Soft Voting,0.587500,0.594200,0.565400,0.099500
4,1.2D,ProtBERT,truncated_1022,Logistic Regression,0.637076,0.642279,0.601449,0.194298
